In [1]:
import os, sys

# Make sure we're in the project root
os.chdir("/Users/danesh/Documents/GitHub/trueQ")

# Ensure project root is on sys.path
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from config import *
from noise_models import *

In [2]:
import trueq as tq
from trueq import Gate
import random
import numpy as np
import trueq.simulation as tqs

In [8]:
cnot_gate_rotation = 0.073675/2
x_strength = 0.005

In [17]:
zxzxz_dict = {gate: ZXZXZ_decompose(tq.Circuit([{0:gate}])) for gate in clifford_gates}
easy_gate_circuits = [ZXZXZ_decompose(tq.Circuit([{0:easy}])) for easy in easy_gates]


sim = tq.Simulator()
        
noisy_sim = (
    sim.add_overrotation(single_sys=x_strength, match=tqs.GateMatch(tq.Gate.sx))
        .add_overrotation(single_sys = cnot_gate_rotation, multi_sys=cnot_gate_rotation, match=match_cnot)
)
infidelity = 1 - sum([process_fidelity(circ, noisy_sim) for circ in easy_gate_circuits]) / len(easy_gate_circuits)

zxzxz_simulator = noisy_sim
noisy_easy_gates = {gate: tq.Gate(zxzxz_simulator.operator(zxzxz_dict[gate]).mat()) for gate in clifford_gates}

def gate_replace(gate):
    return noisy_easy_gates[gate]

noisy_sim_cb = (
    tq.Simulator().add_gate_replace(gate_replace, match=tqs.GateMatch(clifford_gates))
        .add_overrotation(single_sys = cnot_gate_rotation, multi_sys=cnot_gate_rotation, match=match_cnot)
)
 
noisy_sim2 =(
    tq.Simulator().add_overrotation(single_sys=x_strength, match=tqs.GateMatch(tq.Gate.sx))
        .add_overrotation(single_sys = cnot_gate_rotation, multi_sys=cnot_gate_rotation, match=match_cnot)
)

In [13]:
def run_cb_on_cycleZXZXZ(cycle: tq.Cycle, n_decays, n_randomizations, m_vals, noisy_sim, twirl = 'P', propagate_correction=False):
    
    cb_circuits = tq.make_cb(cycles= cycle, n_circuits= n_randomizations, n_random_cycles= m_vals, n_decays=n_decays, twirl=twirl, propagate_correction=propagate_correction)
    cb_circuits2 = transpiler.compile(cb_circuits)
    noisy_sim.run(cb_circuits2, n_shots=np.inf)
    return calculate_process_fidelity(cb_circuits2)

In [14]:
cnot_circ = tq.Cycle({(0,1): Gate.cx})

In [15]:
# Run CNOT gate CB 10 times
cnot_results = []
print("Running CNOT gate CB 10 times...")
for i in range(10):
    print(f"  Run {i+1}/10...")
    cnot_cb_result = run_cb_on_cycleZXZXZ(cnot_circ, n_decays=16, n_randomizations=200, m_vals=[4,20,40,80], noisy_sim=noisy_sim2, twirl=tq.Twirl({(0,): 'U', (1,): 'U'}), propagate_correction=True)
    cnot_results.append(cnot_cb_result)
    print(f"    Result: {cnot_cb_result}")
cnot_average = np.mean(cnot_results)
cnot_std = np.std(cnot_results)
print(f"\nCNOT gate CB results: {cnot_results}")
print(f"CNOT gate CB average infidelity: {cnot_average}")
print(f"CNOT gate CB std dev: {cnot_std}")

Running CNOT gate CB 10 times...
  Run 1/10...
    Result: 0.9977738057765111
  Run 2/10...
    Result: 0.9977602012938248
  Run 3/10...
    Result: 0.9976967935631134
  Run 4/10...
    Result: 0.9975085549309824
  Run 5/10...
    Result: 0.9976152015280266
  Run 6/10...
    Result: 0.9977901366650106
  Run 7/10...
    Result: 0.997599362219445
  Run 8/10...
    Result: 0.9976073978137651
  Run 9/10...
    Result: 0.9976404083125132
  Run 10/10...
    Result: 0.9975749371005854

CNOT gate CB results: [0.9977738057765111, 0.9977602012938248, 0.9976967935631134, 0.9975085549309824, 0.9976152015280266, 0.9977901366650106, 0.997599362219445, 0.9976073978137651, 0.9976404083125132, 0.9975749371005854]
CNOT gate CB average infidelity: 0.9976566799203779
CNOT gate CB std dev: 8.95416752996837e-05


In [23]:
circs = tq.make_cb(cnot_circ, n_decays=16, n_circuits=200, n_random_cycles=[4,20,40,80], twirl=tq.Twirl({(0,): 'U', (1,): 'U'}), propagate_correction=True)

In [24]:
circs[0].draw()

DisplayWrapper(<svg xmlns="http://w...)

In [16]:
# Run CNOT gate CB 10 times
i_results = []
print("Running i gate CB 10 times...")
for i in range(10):
    print(f"  Run {i+1}/10...")
    i_cb_result = run_cb_on_cycleZXZXZ(cycle={}, n_decays=4, n_randomizations=200, m_vals=[8,40,80,120], noisy_sim=noisy_sim_cb, twirl=tq.Twirl({(0,): 'P'}))
    i_results.append(i_cb_result)
    print(f"    Result: {i_cb_result}")

i_average = np.mean(i_results)
i_std = np.std(i_results)
print(f"\ni gate CB results: {i_results}")
print(f"i gate CB average infidelity: {i_average}")
print(f"i gate CB std dev: {i_std}")

Running i gate CB 10 times...
  Run 1/10...
    Result: 0.9998825276178489
  Run 2/10...
    Result: 0.9998800136627629
  Run 3/10...
    Result: 0.9998773192629016
  Run 4/10...
    Result: 0.9998614710629371
  Run 5/10...
    Result: 0.9998923562322946
  Run 6/10...
    Result: 0.9998814929564602
  Run 7/10...
    Result: 0.9998679467422368
  Run 8/10...
    Result: 0.9998829002197506
  Run 9/10...
    Result: 0.9998819253051761
  Run 10/10...
    Result: 0.9998744894376937

i gate CB results: [0.9998825276178489, 0.9998800136627629, 0.9998773192629016, 0.9998614710629371, 0.9998923562322946, 0.9998814929564602, 0.9998679467422368, 0.9998829002197506, 0.9998819253051761, 0.9998744894376937]
i gate CB average infidelity: 0.9998782442500064
i gate CB std dev: 8.177659250033109e-06


In [14]:
i_predict = 1-noisy_sim_cb.predict_cb(cycle= {}, n_randomizations=10000, twirl=tq.Twirl('P',0)).array('e_F', 'labels').vals[0]
print(f"I gate predict_cb infidelity: {i_predict}")
    
# Run I gate CB 10 times
i_results = []
print("Running I gate CB 10 times...")
for i in range(10):
    print(f"  Run {i+1}/10...")
    i_cb_result = run_cb_on_cycle_identity(n_randomizations=1000, m_vals=[8,40,80,120], noisy_sim=noisy_sim_cb)
    i_results.append(i_cb_result)
    print(f"    Result: {i_cb_result}")

i_average = np.mean(i_results)
i_std = np.std(i_results)
print(f"\nI gate CB results: {i_results}")
print(f"I gate CB average infidelity: {i_average}")
print(f"I gate CB std dev: {i_std}")

I gate predict_cb infidelity: 0.99900489771701
Running I gate CB 10 times...
  Run 1/10...
    Result: 0.9990338237716063
  Run 2/10...
    Result: 0.9989916094234362
  Run 3/10...
    Result: 0.9990192037802814
  Run 4/10...
    Result: 0.9990358341773437
  Run 5/10...
    Result: 0.999019986586491
  Run 6/10...
    Result: 0.9989702914875134
  Run 7/10...
    Result: 0.9990452790260604
  Run 8/10...
    Result: 0.9989945431349204
  Run 9/10...
    Result: 0.9990092325747271
  Run 10/10...
    Result: 0.9990054891550596

I gate CB results: [0.9990338237716063, 0.9989916094234362, 0.9990192037802814, 0.9990358341773437, 0.999019986586491, 0.9989702914875134, 0.9990452790260604, 0.9989945431349204, 0.9990092325747271, 0.9990054891550596]
I gate CB average infidelity: 0.999012529311744
I gate CB std dev: 2.1851508887350866e-05
